# DL · Lecture 13 — Learn attention one step at a time

**Attention lets the decoder look back while it writes.** Start by blending three small vectors. Then discover where the weights come from. The same example appears in the reading and slides.

| Route | What you do | When to use it |
|---|---|---|
| **Part A — understand** | Given weights → weighted sum → query/key scores → softmax → changing focus | First pass in class; about 25–35 minutes with discussion |
| **Part B — train** | Run a provided GRU model and inspect real predictions and alignment | After you can explain Part A |
| **Part C — extend** | Scaling, masks, greedy/beam search and additional exercises | Follow-on practice / HW7 preparation |

**You can stop at the end of Part A and already explain the attention mechanism.** No training, tensor shapes or GRU class is needed for that part. You only need vector addition, scalar multiplication, and a dot product.

Run cells in order within the route. **Run all** also works. Stock Colab has `numpy`, `matplotlib`, and `torch`; no downloads, accounts or API keys are needed. CPU is sufficient. GPU is optional for Part B (Runtime → Change runtime type → T4 GPU). Saved outputs are cleared so you see your own results.

## Part A · 1. Why look back?
Translate `je suis fatigué` → `I am tired`.

The **encoder reads** and makes a vector at each source position. The **decoder writes** one output token at a time. A basic decoder starts from the encoder's final state: one fixed-size summary of the source. That can become a bottleneck for long or detailed inputs.

Picture translating after closing the book. Attention gives the decoder access to the source representations again — **a new look-back at each step**.

**Ask before running anything:** if the decoder is about to produce `tired`, which source position might help? Why might the useful position change for the next word?

We will use three **made-up value vectors**, one per source word: `je=[1,0]`, `suis=[0,1]`, `fatigué=[2,1]`. Their coordinates are just arithmetic features, not literal meanings. This example is not a trained translator.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

words = ["je", "suis", "fatigué"]
values = np.array([[1., 0.], [0., 1.], [2., 1.]])
print("Source information we will blend:")
for word, value in zip(words, values):
    print(f"{word:8s} → {value}")

### 2. Use GIVEN weights first
Suppose the weights for this step are **`[0.2, 0.1, 0.7]`**. They are non-negative and sum to one. Where they came from can wait.

Multiply each value by its weight, then add the results:

```
0.2 × [1,0] = [0.2, 0.0]
0.1 × [0,1] = [0.0, 0.1]
0.7 × [2,1] = [1.4, 0.7]
                 add ↓
context      = [1.6, 0.8]
```

**Predict:** is the answer the vector `[2,1]`, or a blend? Which source position contributes most strongly?

In [ ]:
given_weights = np.array([0.2, 0.1, 0.7])
contributions = given_weights[:, None] * values
context = contributions.sum(axis=0)
print("Contributions:\n", contributions)
print("Context:", context)
assert np.isclose(given_weights.sum(), 1.)
assert np.allclose(context, [1.6, 0.8])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].bar(words, given_weights, color=["#38bdf8", "#a78bfa", "#f59e0b"])
axes[0].set(title="Given weights", ylabel="Share", ylim=(0,1))
for word, value, color in zip(words, values, ["#38bdf8", "#a78bfa", "#f59e0b"]):
    axes[1].scatter(*value, s=80, color=color)
    axes[1].annotate(word, value, xytext=(6,6), textcoords="offset points")
axes[1].scatter(*context, marker="*", s=240, color="black", label="context [1.6, 0.8]")
axes[1].set(xlim=(-.3,2.6), ylim=(-.3,1.5), xlabel="Feature 1", ylabel="Feature 2",
            title="Context is a blend of the values")
axes[1].legend(loc="lower right"); plt.tight_layout(); plt.show()

### Pause · you choose the blend
**Before running:** what will weights `[0,0,1]` produce? What about equal weights?

Edit `your_weights` and predict both coordinates. The first choice selects one value exactly; ordinary softmax with finite scores gives positive weights to every unmasked position. Equal weights give the simple average.

The context is **information for the decoder**, not a word. A separate next-token predictor uses it and the decoder state to predict output vocabulary probabilities.

In [ ]:
your_weights = np.array([0., 0., 1.])  # then try [1/3, 1/3, 1/3]
assert np.all(your_weights >= 0) and np.isclose(your_weights.sum(), 1.)
print("Your context:", your_weights @ values)
# Check after predicting: [2,1] for the first choice; [1, 2/3] for equal weights.

### 3. Where do weights come from? Three jobs

| Role | Plain language | Our example |
|---|---|---|
| **Query** | What am I looking for now? | A vector representing the current decoder state |
| **Key** | What do I compare that request against? | A vector at each source position |
| **Value** | What information do I blend and return? | A vector at each source position |

Here **keys and values use the same vectors**, but their jobs differ. Query `[1,0]` produces these dot-product scores:

```
[1,0] · [1,0] = 1    je
[1,0] · [0,1] = 0    suis
[1,0] · [2,1] = 2    fatigué
```

**Predict:** which position should receive the biggest weight? Are these scores already non-negative shares summing to one? What if a score were negative?

The real model learns its representations. It does not receive the correct next word as the query. We choose vectors by hand here to expose the calculation.

### 4. Scores → softmax → weighted sum
Softmax exponentiates the scores and divides by the total. For `[1,0,2]`, the amounts are approximately `[2.718,1,7.389]`, giving weights `[0.245,0.090,0.665]`.

These **calculated** weights differ slightly from the **given** `[0.2,0.1,0.7]` above. The blending operation is unchanged.

The function below uses NumPy. `scores - max(scores)` keeps exponentiation numerically stable without changing the resulting weights. No scaling, batch dimension or padding mask is needed yet.

In [ ]:
def attend(query, keys, values):
    scores = keys @ query                       # one score per source position
    amounts = np.exp(scores - scores.max())
    weights = amounts / amounts.sum()          # positive shares summing to one
    context = weights @ values                 # weighted blend
    return scores, weights, context

keys = values.copy()
query = np.array([1., 0.])
scores, weights, context = attend(query, keys, values)
for word, score, weight in zip(words, scores, weights):
    print(f"{word:8s} score={score: .1f}   weight={weight:.3f}")
print("Context:", np.round(context, 3))
assert np.allclose(scores, [1., 0., 2.])
assert np.allclose(context, [1.57521038, .75527153])
assert np.isclose(weights.sum(), 1.)

### 5. Change only the query — watch the focus move
Set the new query to `[-1,2]`. The scores become `[-1,2,0]`.

**Predict before running:** which source position gets the largest weight now? Do the values themselves change?

Each decoder step can produce a different query, hence different weights and a fresh context. The side-by-side plots make this visible. These queries are chosen by us; nothing has been trained.

In [ ]:
new_query = np.array([-1., 2.])  # edit this, predict, then rerun
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, q in zip(axes, [np.array([1., 0.]), new_query]):
    s, w, c = attend(q, keys, values)
    ax.bar(words, w, color=["#38bdf8", "#a78bfa", "#f59e0b"])
    ax.set(ylim=(0, 1), ylabel="Attention weight",
           title=f"Query {q.tolist()}\ncontext {np.round(c, 3).tolist()}")
    for i, weight in enumerate(w):
        ax.text(i, weight + .025, f"{weight:.3f}", ha="center")
    print("query", q, "scores", s, "context", np.round(c, 3))
plt.tight_layout(); plt.show()

### 6. Stack the rows: an alignment heatmap
A row describes one decoder step; a column describes one source position. Read across one row first. Brighter means more weight in the context.

The table below is **hand-made**, not obtained by training or by the query examples above. It illustrates possible attention while writing `I / am / tired`. A diagonal is not required: different word orders can produce different patterns.

In [ ]:
illustration = np.array([[.8,.1,.1], [.1,.8,.1], [.2,.1,.7]])
assert np.allclose(illustration.sum(axis=1), 1.)
fig, ax = plt.subplots(figsize=(6,3.7))
im = ax.imshow(illustration, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(3), words)
ax.set_yticks(range(3), ["I", "am", "tired"])
for r in range(3):
    for c in range(3):
        ax.text(c, r, f"{illustration[r,c]:.1f}", ha="center", va="center",
                color="white" if illustration[r,c] > .5 else "black")
ax.set(xlabel="Source position", ylabel="Output step",
       title="Illustration only — not learned weights")
fig.colorbar(im, ax=ax, label="Weight"); plt.tight_layout(); plt.show()

### Stop here · explain attention in your own words
1. Why keep all source states available?
2. Blend our values with weights `[0.1,0.7,0.2]`. What is the context?
3. Complete: query + keys → ___ → ___ → blend values → context.
4. Is attention's `0.7` the probability of generating the word at that source position?
5. Which changes across decoder steps: the query, the source states, or necessarily both?

<details><summary>Reveal checks after discussing</summary>

1. The decoder can consult different source information at different steps.
2. `[0.5,0.9]`.
3. Scores → softmax weights.
4. No. These weights are over **source positions**; a separate predictor produces probabilities over **output tokens**.
5. The decoder state/query can change while the encoded source states stay fixed.

</details>

**One sentence to keep:** attention scores the source for the current query and returns a weighted blend of its information. A heatmap shows those weights, not a causal proof of the final prediction.

You have completed the core explanation. Continue with Part B when ready to see a model learn its own weights.

## Part B · Train a model after understanding the mechanism
Here we train **synthetic sequence reversal**: `3 1 4 → 4 1 3`. This diagnostic has an unambiguous answer and an easy-to-inspect alignment; it is not a translation benchmark.

**Suggested first pass:** run the provided class and training cells, then discuss the examples and heatmap. You do not need to type every GRU line during class. Trace one example before studying the batch shapes.

The reversal task keeps the same number of content tokens. Seq2Seq also supports unequal lengths by decoding until EOS; Part C isolates that stopping mechanism in a small table example.

We fix seeds, but exact results can vary by device/version. Leave `SMOKE=False` for meaningful training; `True` only checks that the pipeline runs. Runtime is printed by the training cell.

In [ ]:
import copy
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

SEED = 13
SMOKE = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(2, torch.get_num_threads()))

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)
CFG = dict(n_train=256 if SMOKE else 2400,
           n_eval=48 if SMOKE else 240,
           epochs=2 if SMOKE else 16,
           batch_size=64, emb=32, hidden=64, lr=0.003)
print("PyTorch:", torch.__version__, "device:", DEVICE, "config:", CFG)


### B1. Get the data — and look at it
Digits `0`–`7` are data tokens; the three special tokens have distinct IDs. Source sequences contain digits only. Targets are `<sos>` + reversed digits + `<eos>`.

We create **disjoint** train/validation/test inputs of length 3–8. An additional test set has lengths 9–12, outside the training range. Validation chooses a checkpoint; test sets are evaluated after training. The longer set probes length generalization, not ordinary held-out performance.

The reversal rule is visible in the data generator because it creates labels. The neural model receives only tensors and must learn the rule from training examples.


In [ ]:
TOKENS = ["<pad>", "<sos>", "<eos>"] + list("01234567")
PAD, SOS, EOS = 0, 1, 2
VOCAB = len(TOKENS)
rng = random.Random(SEED)
seen = set()

def make_examples(n, lo, hi):
    examples = []
    while len(examples) < n:
        src = tuple(rng.randrange(3, VOCAB) for _ in range(rng.randint(lo, hi)))
        if src not in seen:
            seen.add(src)
            examples.append((src, (SOS,) + src[::-1] + (EOS,)))
    return examples

train_data = make_examples(CFG["n_train"], 3, 8)
val_data = make_examples(CFG["n_eval"], 3, 8)
test_data = make_examples(CFG["n_eval"], 3, 8)
long_data = make_examples(CFG["n_eval"], 9, 12)
assert len(seen) == sum(map(len, [train_data, val_data, test_data, long_data]))

def text(ids):
    return " ".join(TOKENS[int(i)] for i in ids)

for src, tgt in train_data[:5]:
    print(text(src), "→", text(tgt))

def collate(examples):
    src, tgt = zip(*examples)
    lengths = torch.tensor([len(s) for s in src])
    return (pad_sequence([torch.tensor(s) for s in src], batch_first=True,
                         padding_value=PAD),
            lengths,
            pad_sequence([torch.tensor(t) for t in tgt], batch_first=True,
                         padding_value=PAD))

def loader(data, shuffle=False):
    return DataLoader(data, batch_size=CFG["batch_size"], shuffle=shuffle,
                      collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

src, lengths, tgt = next(iter(loader(train_data)))
print("Shapes: source", tuple(src.shape), "target", tuple(tgt.shape))
print("One target:     ", text(tgt[0]))
print("Decoder inputs:", text(tgt[0, :-1]))
print("Loss labels:   ", text(tgt[0, 1:]))
assert (tgt[:, 0] == SOS).all()
assert (tgt == EOS).sum().item() == len(tgt)


### B2. Build the encoder–decoder (🔧)
Both variants use the same GRU encoder and GRU decoder dimensions.

- **Fixed context:** initialize the decoder with the encoder's final state; no later access to source states.
- **Attention:** also score every real encoder state at each decoder step, and concatenate the resulting context with the decoder state before predicting a token.

We use **unscaled dot-product attention** here, matching Part A of the reading. Part C adds scaling to Q/K/V. A trainable query projection lets the decoder and encoder states align. Attention adds parameters, so this is an instructional comparison, not a parameter-matched benchmark.

**Padding has two separate fixes:** packing stops the encoder's final state from reading padding; an attention mask prevents assigning weight to padding. Ignoring PAD in the loss does neither of these.


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, use_attention):
        super().__init__()
        self.use_attention = use_attention
        e, h = CFG["emb"], CFG["hidden"]
        self.src_embed = nn.Embedding(VOCAB, e, padding_idx=PAD)
        self.tgt_embed = nn.Embedding(VOCAB, e, padding_idx=PAD)
        self.encoder = nn.GRU(e, h, batch_first=True)
        self.decoder = nn.GRU(e, h, batch_first=True)
        if use_attention:
            self.query = nn.Linear(h, h, bias=False)
        self.output = nn.Linear(2 * h if use_attention else h, VOCAB)

    def encode(self, src, lengths):
        packed = pack_padded_sequence(self.src_embed(src), lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_out, hidden = self.encoder(packed)
        states, _ = pad_packed_sequence(packed_out, batch_first=True,
                                        total_length=src.shape[1])
        return states, hidden, src.ne(PAD)

    def decode(self, tokens, hidden, states, valid):
        # tokens: (B,T), states: (B,S,H), decoded: (B,T,H)
        decoded, hidden = self.decoder(self.tgt_embed(tokens), hidden)
        weights = None
        if self.use_attention:
            scores = self.query(decoded) @ states.transpose(1, 2)
            scores = scores.masked_fill(~valid[:, None, :], float("-inf"))
            weights = scores.softmax(-1)            # (B,T,S)
            context = weights @ states              # (B,T,H)
            decoded = torch.cat([decoded, context], dim=-1)
        return self.output(decoded), hidden, weights

    def forward(self, src, lengths, decoder_inputs):
        states, hidden, valid = self.encode(src, lengths)
        logits, _, weights = self.decode(decoder_inputs, hidden, states, valid)
        return logits, weights

seed_all(SEED)
probe = Seq2Seq(True).to(DEVICE)
with torch.no_grad():
    logits, alignment = probe(src.to(DEVICE), lengths, tgt[:, :-1].to(DEVICE))
assert logits.shape == (*tgt[:, 1:].shape, VOCAB)
assert torch.allclose(alignment.sum(-1), torch.ones_like(alignment.sum(-1)))
assert alignment.masked_select(~src.to(DEVICE).ne(PAD)[:, None, :]).eq(0).all()
print("logits:", tuple(logits.shape), "alignment:", tuple(alignment.shape))
del probe


### B3. Train with teacher forcing (🔧)
The decoder sees the **true previous token** (`tgt[:, :-1]`) and predicts the next (`tgt[:, 1:]`). It never sees the current label as its input at the same step.

For speed, a teacher-forced sequence is passed to the GRU in one call. The GRU still computes states recurrently. We average cross-entropy over **non-padding target tokens**, including EOS, and clip gradients to 1.0. Both models get the same examples, epochs, batch order and learning rate.

The best **validation teacher-forced loss** selects the checkpoint. This loss is not free-running accuracy: at inference the decoder must consume its own predictions.


In [ ]:
loss_sum = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")

def loss_epoch(model, batches, optimizer=None):
    model.train(optimizer is not None)
    total_loss, total_tokens = 0., 0
    for src, lengths, tgt in batches:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        labels = tgt[:, 1:]
        count = labels.ne(PAD).sum().item()
        with torch.set_grad_enabled(optimizer is not None):
            logits, _ = model(src, lengths, tgt[:, :-1])
            loss = loss_sum(logits.reshape(-1, VOCAB), labels.reshape(-1))
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                (loss / count).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item()
        total_tokens += count
    return total_loss / total_tokens

models, histories = {}, {}
started = time.perf_counter()
for name, use_attention in [("fixed context", False), ("attention", True)]:
    seed_all(SEED)
    model = Seq2Seq(use_attention).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    train_batches, val_batches = loader(train_data, True), loader(val_data)
    history, best_loss, best_state = [], float("inf"), None
    print("\n", name, "parameters:", sum(p.numel() for p in model.parameters()))
    for epoch in range(1, CFG["epochs"] + 1):
        train_loss = loss_epoch(model, train_batches, optimizer)
        val_loss = loss_epoch(model, val_batches)
        history.append((train_loss, val_loss))
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
        if epoch == 1 or epoch % 4 == 0 or epoch == CFG["epochs"]:
            print(f"epoch {epoch:2d}: train {train_loss:.3f}, val {val_loss:.3f}")
    model.load_state_dict(best_state)
    models[name], histories[name] = model.eval(), history
print(f"Training elapsed: {time.perf_counter() - started:.1f}s")

for name, history in histories.items():
    h = np.array(history)
    plt.plot(range(1, len(h)+1), h[:, 1], label=name + " (validation)")
plt.xlabel("Epoch"); plt.ylabel("Teacher-forced NLL / token")
plt.legend(); plt.show()


### B4. Free-running inference — no target tokens allowed
Start from SOS, predict one token, and feed that prediction back. Stop each sequence at EOS; use `max_steps` as a safety cap. PAD and SOS are not legal generated tokens here.

The function below takes **only source inputs**, not target answers or target lengths. It returns attention rows from the actual free-running path, rather than a teacher-forced visualization.


In [ ]:
@torch.no_grad()
def generate(model, src, lengths, max_steps=16):
    if max_steps < 1:
        raise ValueError("max_steps must be positive")
    model.eval()
    src = src.to(DEVICE)
    states, hidden, valid = model.encode(src, lengths)
    token = torch.full((len(src), 1), SOS, dtype=torch.long, device=DEVICE)
    done = torch.zeros(len(src), dtype=torch.bool, device=DEVICE)
    tokens, rows = [], []
    for _ in range(max_steps):
        logits, hidden, weights = model.decode(token, hidden, states, valid)
        scores = logits[:, -1].clone()
        scores[:, [PAD, SOS]] = float("-inf")
        nxt = scores.argmax(-1)
        nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
        tokens.append(nxt.cpu())
        if weights is not None:
            rows.append(weights[:, 0].cpu())
        done |= nxt.eq(EOS)
        token = nxt[:, None]
        if bool(done.all()):
            break
    return torch.stack(tokens, 1), (torch.stack(rows, 1) if rows else None)

def through_eos(ids):
    result = []
    for token in ids:
        token = int(token)
        result.append(token)
        if token == EOS:
            break
    return result

@torch.no_grad()
def evaluate(model, data):
    exact, token_hits, token_total, ended = 0, 0, 0, 0
    by_length = {}
    for src, lengths, tgt in loader(data):
        predictions, _ = generate(model, src, lengths)
        for p, gold, length in zip(predictions.tolist(), tgt[:, 1:].tolist(), lengths.tolist()):
            p, gold = through_eos(p), through_eos(gold)
            ok = p == gold  # includes EOS: truncation is not a correct answer
            exact += int(ok)
            ended += int(EOS in p)
            token_hits += sum(a == b for a, b in zip(p, gold))
            token_total += max(len(p), len(gold))  # penalize missing AND extra tokens
            group = by_length.setdefault(length, [0, 0])
            group[0] += int(ok); group[1] += 1
    return dict(exact=exact/len(data), token=token_hits/token_total,
                ended=ended/len(data), by_length=by_length)

results = {}
for name, model in models.items():
    for split, data in [("held-out 3–8", test_data), ("longer 9–12", long_data)]:
        r = evaluate(model, data)
        results[name, split] = r
        print(f"{name:13s} | {split:13s} | exact {r['exact']:.1%} | "
              f"position accuracy incl. EOS {r['token']:.1%} | emitted EOS {r['ended']:.1%}")

for name in models:
    groups = {**results[name, "held-out 3–8"]["by_length"],
              **results[name, "longer 9–12"]["by_length"]}
    xs = sorted(groups)
    plt.plot(xs, [groups[x][0]/groups[x][1] for x in xs], "o-", label=name)
plt.axvline(8.5, color="gray", linestyle="--", label="outside training lengths")
plt.xlabel("Source length"); plt.ylabel("Exact sequence accuracy (includes EOS)")
plt.ylim(-.05, 1.05); plt.legend(); plt.show()


**Interpret your result, not a promised result.** Compare the two models on held-out lengths, then on longer sequences. Attention makes source states accessible; it does **not** guarantee length extrapolation, perfect accuracy, or superiority on every seed. A short reversal task can also be solved using a fixed context.

If loss falls but exact accuracy stays low, inspect predictions: a single wrong digit, early EOS or failure to stop makes an entire sequence incorrect. To tune epochs or dimensions, use validation data; keep the held-out tests for your final comparison.


In [ ]:
# A fixed slice, not a search for the nicest-looking examples.
for src_ids, target in test_data[:6]:
    src, lengths, _ = collate([(src_ids, target)])
    print("\nsource:", text(src_ids), " | gold:", text(target[1:]))
    for name, model in models.items():
        pred, _ = generate(model, src, lengths)
        print(f"  {name:13s}:", text(through_eos(pred[0].tolist())))


### B5. Look at the learned soft alignment
For reversal, an ideal position alignment runs from the **rightmost** source digit to the leftmost. Repeated digits can make alternative alignments plausible; EOS has no one-to-one source digit. The plot uses a fixed held-out example and shows generated tokens, even if they are wrong.

Each row sums to one across real source positions. Attention weights describe where this context was drawn from; they are **not a causal explanation** of the entire model.


In [ ]:
example_index = 0  # change this to inspect another example
src_ids, target = test_data[example_index]
src, lengths, _ = collate([(src_ids, target)])
pred, rows = generate(models["attention"], src, lengths)
output_ids = through_eos(pred[0].tolist())
heatmap = rows[0, :len(output_ids), :len(src_ids)].numpy()
assert np.allclose(heatmap.sum(axis=1), 1., atol=1e-5)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(heatmap, aspect="auto", vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(src_ids)), [TOKENS[i] for i in src_ids])
ax.set_yticks(range(len(output_ids)), [TOKENS[i] for i in output_ids])
ax.set_xlabel("Source position / digit"); ax.set_ylabel("Generated token")
ax.set_title("Free-running attention (gold: " + text(target[1:]) + ")")
fig.colorbar(im, ax=ax, label="Attention weight"); plt.tight_layout(); plt.show()


## Part C · Extensions for a second pass
Keep this part for after you can explain the blend and read the learned heatmap. Scaling and masks are implementation refinements; beam search is a decoding strategy. They are useful HW7 preparation, but are not prerequisites for understanding Part A.

### C1. Query, Key, Value — add scaling and a padding mask
Here is the reading's scaled example (Part C), now computed from actual Q, K, V vectors. The function accepts one or many queries. A `True` mask position is a real key; padding is masked **before** softmax.

Every query needs at least one unmasked key. Otherwise softmax of all negative infinity is undefined. The context dimension comes from **V**, while the score scaling uses the dimension shared by **Q and K**.


In [ ]:
def scaled_attention(Q, K, V, valid=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if valid is not None:
        if not bool(valid.any(dim=-1).all()):
            raise ValueError("Each sequence must have at least one valid key")
        scores = scores.masked_fill(~valid.unsqueeze(-2), float("-inf"))
    weights = scores.softmax(dim=-1)
    return weights @ V, weights

Q = torch.tensor([[1., 0.]])
K = torch.eye(2)
V = 10 * torch.eye(2)
c, a = scaled_attention(Q, K, V)
print("Scaled weights:", a, "context:", c)
assert torch.allclose(c, torch.tensor([[6.6976, 3.3024]]), atol=1e-4)
# A padded key with a huge score must get zero weight.
K_pad = torch.cat([K, torch.tensor([[100., 100.]])])
V_pad = torch.cat([V, torch.tensor([[999., 999.]])])
c_pad, a_pad = scaled_attention(Q, K_pad, V_pad,
                                torch.tensor([True, True, False]))
assert a_pad[0, -1].item() == 0
assert torch.allclose(c, c_pad)
print("Masked weights:", a_pad)


### Why divide by √d?
For independent, zero-mean, unit-variance query/key components, the dot-product variance grows approximately like `d`. Dividing by √d keeps it near one. Real learned vectors need not satisfy those assumptions exactly.

Run the experiment and compare score standard deviation and attention entropy. Low entropy means a sharper distribution; this is a diagnostic, not a guarantee that training will fail.


In [ ]:
g = torch.Generator().manual_seed(SEED)
print("dim | raw std | scaled std | raw entropy | scaled entropy")
for d in [4, 64, 512]:
    q = torch.randn(256, 1, d, generator=g)
    k = torch.randn(256, 12, d, generator=g)
    raw = (q @ k.transpose(-2, -1)).squeeze(1)
    scaled = raw / math.sqrt(d)
    def entropy(s):
        p = s.softmax(-1)
        return -(p * p.clamp_min(1e-12).log()).sum(-1).mean().item()
    print(f"{d:3d} | {raw.std():7.2f} | {scaled.std():10.2f} | "
          f"{entropy(raw):11.2f} | {entropy(scaled):14.2f}")


### C2. Different lengths, stopping, and beam search
Reversal keeps the same number of content tokens. To isolate **decoding mechanics**, use a tiny hand-specified conditional model below: a four-word source can correspond to a three-word output. This table is **not** a trained translation model.

The first table reproduces the lecture's search example: greedy chooses `the cat` with joint probability `0.5 × 0.7 = 0.35`; width-2 beam finds `a dog` with `0.4 × 0.9 = 0.36`. Every row is a normalized conditional distribution. EOS transitions here have probability one.

Search sums **log-probabilities**, including EOS. Ended hypotheses are retained without expanding them. We keep the top `width` candidates; finite-width beam search is approximate and is not guaranteed to find the globally best sequence. For this tiny equal-length example, exact enumeration is easy.


In [ ]:
NEXT = {
    "<sos>": {"the": .5, "a": .4, "my": .1},
    "the": {"cat": .7, "dog": .3},
    "a": {"dog": .9, "cat": .1},
    "my": {"cat": 1.},
    "cat": {"<eos>": 1.}, "dog": {"<eos>": 1.},
}

def beam_search(table, width=1, max_steps=10, alpha=0.):
    if width < 1 or max_steps < 1:
        raise ValueError("width and max_steps must be positive")
    for distribution in table.values():
        if (not distribution or any(p <= 0 for p in distribution.values())
                or not math.isclose(sum(distribution.values()), 1.)):
            raise ValueError("Each row must be a positive probability distribution")
    beams = [([], 0., False)]  # tokens, summed log-probability, ended
    def rank(item):
        tokens, logp, _ = item
        return logp / max(1, len(tokens)) ** alpha
    for _ in range(max_steps):
        candidates = []
        for tokens, logp, ended in beams:
            if ended:
                candidates.append((tokens, logp, True))
                continue
            previous = tokens[-1] if tokens else "<sos>"
            for token, p in table[previous].items():
                candidates.append((tokens + [token], logp + math.log(p), token == "<eos>"))
        beams = sorted(candidates, key=rank, reverse=True)[:width]
        if all(ended for _, _, ended in beams):
            break
    # Report ended=False when the cap is reached; do not silently call it complete.
    return max(beams, key=rank)

for width in [1, 2, 3]:
    tokens, logp, ended = beam_search(NEXT, width)
    print("width", width, tokens, "joint", round(math.exp(logp), 3), "ended", ended)
assert beam_search(NEXT, 1)[0] == ["the", "cat", "<eos>"]
assert beam_search(NEXT, 2)[0] == ["a", "dog", "<eos>"]
assert beam_search(NEXT, 3)[0] == beam_search(NEXT, 2)[0]

translation_table = {"<sos>": {"I": 1.}, "I": {"am": 1.},
                     "am": {"tired": 1.}, "tired": {"<eos>": 1.}}
print("Four source tokens: je suis très fatigué")
print("Three output words (toy summary):", beam_search(translation_table)[0])
loop_table = copy.deepcopy(translation_table)
loop_table["am"] = {"I": .9, "tired": .1}
loop_output = beam_search(loop_table, max_steps=8)
print("Loop capped safely:", loop_output)
assert not loop_output[2] and len(loop_output[0]) == 8


### Length normalization changes the objective
Adding a token contributes a non-positive log-probability, but **different sequences have different conditional probabilities**: a longer sequence can still outrank a shorter one. Raw log-probability often favors shorter outputs. A length penalty is a heuristic, not a universal correction.

Our optional `alpha=1` ranks by average log-probability per generated token (including EOS); `alpha=0` uses the raw sum. The example below flips the ranking. A cap-truncated hypothesis remains unfinished even if its normalized score is high. For a real decoder, also define how to select among completed hypotheses and evaluate the resulting text, rather than assuming a higher score means a better answer.


In [ ]:
short_logp = math.log(.45)
long_logp = math.log(.40)
print("Raw score:", {"short (2 tokens)": short_logp, "long (5 tokens)": long_logp})
print("Mean score:", {"short": short_logp/2, "long": long_logp/5})
assert short_logp > long_logp and short_logp/2 < long_logp/5
print("Product underflow:", 0.01 ** 200, " | stable log score:", 200 * math.log(.01))


### C3. Further practice — predict first, then edit the scaffold
**P1 — change the query.** Predict the weights for `Q=[0,1]` before running. Then try `[1,1]`. What changes in the context? Why doesn't the result pick just one value?


In [ ]:
practice_Q = torch.tensor([[0., 1.]])  # TODO: also try [1., 1.]
practice_c, practice_a = scaled_attention(practice_Q, torch.eye(2), 10*torch.eye(2))
print("weights", practice_a, "context", practice_c)


**P2 — duplicate a key AND its value.** Identical keys receive equal individual weights. Does the **total mass** on that pair stay the same as the original key's weight? Does the context stay unchanged? Predict before running. Add an assertion checking that the two duplicated positions have equal weights.


In [ ]:
base_K = torch.eye(2)
base_V = 10 * torch.eye(2)
dup_K = torch.cat([base_K, base_K[:1]])
dup_V = torch.cat([base_V, base_V[:1]])
base_c, base_a = scaled_attention(torch.tensor([[1., 0.]]), base_K, base_V)
dup_c, dup_a = scaled_attention(torch.tensor([[1., 0.]]), dup_K, dup_V)
print("original:", base_a, base_c)
print("duplicated:", dup_a, dup_c)
# TODO: compare dup_a[0, 0] and dup_a[0, 2], and compare their sum with base_a[0, 0].


**P3 — trace teacher forcing.** For source `2 0 5`, write the decoder inputs and labels including SOS/EOS. At inference, if the first generated token is wrong, what becomes the second decoder input? Explain why a good teacher-forced loss does not guarantee a correct free-running sequence.

**P4 — test your stopping rule.** Edit `loop_table` so greedy always returns to `I`. Increase the cap. Does the program still terminate? Keep `ended=False` for a capped, unfinished output.

**P5 — greedy ties beam.** Change the probabilities after `the` so width 1 and width 2 both prefer `the cat`. Keep every row normalized. Explain why a larger beam can tie a smaller one.


In [ ]:
practice_table = copy.deepcopy(NEXT)
# TODO: change both probabilities in practice_table["the"] (they must sum to one).
for width in [1, 2]:
    print(width, beam_search(practice_table, width))


**P6 — diagnose the bottleneck.** Draw four source states and three decoder steps. For the fixed-context model, mark the only source-to-decoder initialization. For attention, draw the additional path from **all four** states into the context at step 2. Then inspect the accuracy-by-length plot. What does this one run support, and what does it not establish?

**P7 — a small experiment.** Before changing `hidden`, training length, or epochs, predict an outcome. Retrain **both** models with the same budget, use validation to choose settings, then report final test exact accuracy, EOS rate, parameter counts, and runtime. If you repeatedly inspect the test set while tuning, make a fresh final test set. Do not claim attention always wins from one seed.

**P8 — HW7 bridge.** Reimplement `scaled_attention` using NumPy, including the padding mask. Compare it to the worked Q/K/V result `[6.6976, 3.3024]`. Draw the matrix shapes: Q `(T,D)`, K `(S,D)`, V `(S,Dv)` → weights `(T,S)` → contexts `(T,Dv)`.


<details>
<summary>Checks and hints — open after trying</summary>

- P1: `[0,1]` swaps the original weights and context coordinates. `[1,1]` gives equal scores and context `[5,5]`.
- P2: duplicate keys tie, but the pair's total mass increases because softmax's denominator also changes. The context generally shifts toward the duplicated value; it is not simply the original weight split in two.
- P3: inputs are `<sos> 5 0 2`, labels are `5 0 2 <eos>`. At inference the next input is the model's own first prediction.
- P4: the cap guarantees finite execution, not a completed or meaningful output.
- P5: `the → cat=.9, dog=.1` makes `the cat` score `.45`, above `.36` for `a dog`.
- P6–7: attention adds direct source access, but optimization, capacity, data, and decoding still matter. Report observations without inventing an alignment or a guaranteed accuracy advantage.
- P8: stable softmax subtracts the row maximum before exponentiation; mask invalid keys before this operation and reject all-masked rows.

</details>

## What to take into Lecture 14
The encoder reads; the decoder generates until EOS; teacher forcing supplies the true previous token only during training. Attention constructs a new weighted context at every step, and its heatmap is a soft alignment. This model **still has RNNs**. Transformers use attention with positional information and causal masking to build sequence models without recurrent state updates.

**Further reading:** the course Lecture 13 reading and slide deck; Lecture 14 for self-attention; HW7 for implementing attention and decoding yourself.
